# 🧱 Part 06: Architecture Refinements: RMSNorm, SwiGLU, RoPE, Pre-Norm

> **Previous context**: Mini-GPT gave us a working decoder. Modern models refine the same skeleton for stability and efficiency.
> **Goal for this part**: Implement common LLaMA-style refinements and see what problem each one solves.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why refine the block

A basic Transformer works, but deep training needs stable normalization, expressive feed-forward layers, and position handling that extrapolates better.

## 1. RMSNorm

RMSNorm normalizes by root mean square. It keeps the scale controlled with less machinery than LayerNorm.

## 2. SwiGLU

SwiGLU gives the feed-forward layer a learned gate, so the model can choose which features pass through.

## 3. RoPE and Pre-Norm

RoPE rotates query/key vectors to encode relative position. Pre-Norm places normalization before sublayers to improve training stability.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] RMSNorm controls scale.
- [ ] SwiGLU adds a gate to the feed-forward path.
- [ ] RoPE injects position into attention through rotations.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [2]:
# Teaching note: follow this line to see the main step.

class FeedForward_Old(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock_Old(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # Teaching note: follow this line to see the main step.
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Teaching note: follow this line to see the main step.
        x = self.norm1(x + self.attention(x, x, x, need_weights=False)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.```
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.5. Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [3]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Teaching note: follow this line to see the main step.
mu = x.mean()
print(f"Read the values printed above and connect them to the concept in this cell.{mu:.1f}")

# Teaching note: follow this line to see the main step.
var = torch.mean((x - mu) ** 2)
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"        = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Teaching note: follow this line to see the main step.
sigma = torch.sqrt(var)
print(f"Read the values printed above and connect them to the concept in this cell.{var:.1f} = {sigma:.4f}")

# Teaching note: follow this line to see the main step.
x_norm = (x - mu) / sigma
print(f"Read the values printed above and connect them to the concept in this cell.{sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"         x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"Read the values printed above and connect them to the concept in this cell.{[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"Read the values printed above and connect them to the concept in this cell.{x_norm.mean():.4f}Read the values printed above and connect them to the concept in this cell.{x_norm.std(unbiased=False):.4f} (=1)")

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
print(f"Result:")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.        = (9 + 1 + 1 + 9)/4 = 5.0
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.         x[0] = (1.0 - 4) / 2.2361 = -1.3416
         x[1] = (3.0 - 4) / 2.2361 = -0.4472
         x[2] = (5.0 - 4) / 2.2361 =  0.4472
         x[3] = (7.0 - 4) / 2.2361 =  1.3416
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
RMS(x) = √( (x₁² + x₂² + x₃² + x₄²) / 4 )

RMSNorm(x) = x / RMS(x) × γ
```

Read the values printed above and connect them to the concept in this cell.```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [4]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Teaching note: follow this line to see the main step.
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"          = (1+9+25+49)/4")
print(f"          = {84/4:.1f}")
print(f"    RMS = √{mean_sq:.1f} = {rms:.4f}")
print()

# Teaching note: follow this line to see the main step.
x_rmsnorm = x / rms
print(f"Step 2 — RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"         x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print(f"\nRMSNorm Result: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# Teaching note: follow this line to see the main step.
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print(f"Read the values printed above and connect them to the concept in this cell.{rms_after:.4f}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{x_rmsnorm.mean():.4f}）")
print(f"Read the values printed above and connect them to the concept in this cell.")
print()

# Teaching note: follow this line to see the main step.
ln = nn.LayerNorm(4, elementwise_affine=False)  # Teaching note: follow this line to see the main step.
x_ln = ln(x)
print(f"LayerNorm Result: {[f'{v:.4f}' for v in x_ln.tolist()]}Read the values printed above and connect them to the concept in this cell.{x_ln.mean():.4f}  std={x_ln.std(unbiased=False):.4f}")
print(f"RMSNorm  Result: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}Read the values printed above and connect them to the concept in this cell.{x_rmsnorm.mean():.4f}  RMS={rms_after:.4f}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.          = (1+9+25+49)/4
          = 21.0
    RMS = √21.0 = 4.5826

Step 2 — RMSNorm = x / 4.5826
         x[0] = 1.0 / 4.5826 = 0.2182
         x[1] = 3.0 / 4.5826 = 0.6547
         x[2] = 5.0 / 4.5826 = 1.0911
         x[3] = 7.0 / 4.5826 = 1.5275

RMSNorm Result: ['0.2182', '0.6547', '1.0911', '1.5275']
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [5]:
# Teaching note: follow this line to see the main step.

class RMSNorm(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # Teaching note: follow this line to see the main step.
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # Teaching note: follow this line to see the main step.
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma


# Teaching note: follow this line to see the main step.
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # Teaching note: follow this line to see the main step.
out_rms = rmsn(batch)
out_ln = ln(batch)

print('Read the values printed above and connect them to the concept in this cell.')
print(f"Read the values printed above and connect them to the concept in this cell.{batch.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{out_rms.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# Teaching note: follow this line to see the main step.
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"Read the values printed above and connect them to the concept in this cell.{d_model}) + β({d_model}) = {ln_params}")
print(f"Read the values printed above and connect them to the concept in this cell.{d_model}) = {rmsn_params}")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
FFN(x) = W₂ · ReLU(W₁ · x)

Exercise passed: you have understood this step.```

Read the values printed above and connect them to the concept in this cell.

In [6]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"Result:{[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print('Read the values printed above and connect them to the concept in this cell.')
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = 'Read the values printed above and connect them to the concept in this cell.' if h >= 0 else 'Read the values printed above and connect them to the concept in this cell.'
    print(f"Read the values printed above and connect them to the concept in this cell.{i}: {h:6.2f} → {r:6.2f}  {action}")

print(f"Read the values printed above and connect them to the concept in this cell.{sum(1 for h in hidden if h < 0)}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printe

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
- Exercise passed: you have understood this step.
- Read the values printed above and connect them to the concept in this cell.  - Read the values printed above and connect them to the concept in this cell.  - Read the values printed above and connect them to the concept in this cell.  - Read the values printed above and connect them to the concept in this cell.
Exercise passed: you have understood this step.
Read the values printed above and connect them to the concept in this cell.
```
SwiGLU(x) = (W_up · x)  ⊙  Swish(W_gate · x)
Read the values printed above and connect them to the concept in this cell.              
Swish(a) = a × sigmoid(a) = a / (1 + e^(-a))
```

Read the values printed above and connect them to the concept in this cell.

In [7]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {"'Note'"}")
print("-" * 52)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))  # x * sigmoid(x)
    note = ""
    if x < 0:
        note = f"Read the values printed above and connect them to the concept in this cell.{swish:.4f}"
    elif x == 0:
        note = 'Read the values printed above and connect them to the concept in this cell.'
    else:
        note = 'Exercise passed: you have understood this step.'
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print('"Key observation："')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
x ReLU(x) Swish(x) Note----------------------------------------------------
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Exercise passed: you have understood this step.Exercise passed: you have understood this step.Exercise passed: you have understood this step.Exercise passed: you have understood this step.
Key observation:  1. Read the values printed above and connect them to the concept in this cell.  2. Read the values printed above and connect them to the concept in this cell.  3. Read the values printed above and connect them to the concept in this cell.
Read the values printed abo

In [8]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
d_model = 4
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

# Teaching note: follow this line to see the main step.
up = W_up(x)
# Teaching note: follow this line to see the main step.
gate = F.silu(W_gate(x))  # silu = Swish
# Teaching note: follow this line to see the main step.
gated = up * gate
# Teaching note: follow this line to see the main step.
out = W_down(gated)

print(f"Input x: {x.tolist()}")
print()
print(f"Read the values printed above and connect them to the concept in this cell.{[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"Read the values printed above and connect them to the concept in this cell.{[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"Read the values printed above and connect them to the concept in this cell.{[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"Output (W_down·gated):    {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Exercise passed: you have understood this step.')
print('Exercise passed: you have understood this step.')

Read the values printed above and connect them to the concept in this cell.
Input x: [[1.0, -0.5, 2.0, 0.30000001192092896]]
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Output (W_down·gated): ['-0.034', '0.101', '0.114', '-0.059']
Read the values printed above and connect them to the concept in this cell.Exercise passed: you have understood this step.Exercise passed: you have understood this step.

In [9]:
# Teaching note: follow this line to see the main step.

class FeedForward_SwiGLU(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff ≈ 2 * d_model * 4d_model
            # → d_ff ≈ 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # Teaching note: follow this line to see the main step.
            d_ff = max(d_ff, d_model)  # Teaching note: follow this line to see the main step.
        
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))


# Teaching note: follow this line to see the main step.
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

print('Read the values printed above and connect them to the concept in this cell.')
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}×{4*d_model}) + W2({4*d_model}×{d_model})")
print(f"           = {d_model*4*d_model:,} + {4*d_model*d_model:,}")
print(f"           = {old_p:,}Read the values printed above and connect them to the concept in this cell.")
print()
d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256
print(f"SwiGLU FFN: W_gate({d_model}×{d_ff_s}) + W_up({d_model}×{d_ff_s}) + W_down({d_ff_s}×{d_model})")
print(f"           = {d_model*d_ff_s:,} + {d_model*d_ff_s:,} + {d_ff_s*d_model:,}")
print(f"           = {new_p:,}Read the values printed above and connect them to the concept in this cell.")
print()
print(f"Read the values printed above and connect them to the concept in this cell.{new_p/old_p:.2f}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.d_model = 512

ReLU FFN:  W1(512×2048) + W2(2048×512)
           = 1,048,576 + 1,048,576
Read the values printed above and connect them to the concept in this cell.
SwiGLU FFN: W_gate(512×1536) + W_up(512×1536) + W_down(1536×512)
           = 786,432 + 786,432 + 786,432
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```python
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
x ──→ Attention ──→ + ──→ LayerNorm ──→ Output  │                    ↑
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Exercise passed: you have understood this step.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```python
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
```
x ──→ LayerNorm ──→ Attention ──→ + ──→ Output  │                                   ↑
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [10]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
d_model = 4
num_layers = 8  # Teaching note: follow this line to see the main step.

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# Teaching note: follow this line to see the main step.
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

# Teaching note: follow this line to see the main step.
x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

# Forward + Backward
loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

# Teaching note: follow this line to see the main step.
print(f"Read the values printed above and connect them to the concept in this cell.{num_layers}Read the values printed above and connect them to the concept in this cell.")
print()
print(f"{'Read the values printed above and connect them to the concept in this cell.':>6s}  {'Read the values printed above and connect them to the concept in this cell.':>16s}  {'Read the values printed above and connect them to the concept in this cell.':>16s}")
print("-" * 42)

for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

# Teaching note: follow this line to see the main step.
grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{grad_post_first:.6f}")
print(f"Read the values printed above and connect them to the concept in this cell.{grad_pre_first:.6f}")
print(f"  Pre-Norm / Post-Norm = {grad_pre_first/grad_post_first:.2f}x")
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.------------------------------------------
Layer 1:          0.411592          6.629198
Layer 2:          0.319549          5.358179
Layer 3:          0.309842          6.145226
Layer 4:          0.317241          5.256918
Layer 5:          0.202899          3.324109
Layer 6:          0.217564          2.635670
Layer 7:          0.114408          3.379233
Layer 8:          0.532629          4.577710

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Pre-Norm / Post-Norm = 16.11x

Read the values printed above and connect them to the concept in this cell.Read the values printed above and co

#### Read the values printed above and connect them to the concept in this cell.
- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

---

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
LLaMA Block = Pre-Norm (RMSNorm) + Attention + Pre-Norm (RMSNorm) + SwiGLU FFN

  x
  │
Read the values printed above and connect them to the concept in this cell.  │                           │
  ├───────────────────────────┘
  │
Read the values printed above and connect them to the concept in this cell.  │                            │
  └────────────────────────────┘
  ↓
Output```

In [11]:
class LLaMABlock(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Teaching note: follow this line to see the main step.
        self.norm_attn = RMSNorm(d_model)
        # Teaching note: follow this line to see the main step.
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Teaching note: follow this line to see the main step.
        self.norm_ffn = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # Teaching note: follow this line to see the main step.
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out  # Teaching note: follow this line to see the main step.
        
        h = self.norm_ffn(x)
        x = x + self.ffn(h)  # Teaching note: follow this line to see the main step.
        
        return x

# Teaching note: follow this line to see the main step.
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print('"=== LLaMA Block test ==="')
print(f"Read the values printed above and connect them to the concept in this cell.{test_x.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{out_new.shape}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

=== LLaMA Block test ===Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.      Post-Norm + LayerNorm + ReLU FFN
Read the values printed above and connect them to the concept in this cell.
2019  GPT-2 / GPT-3
      Pre-Norm + LayerNorm + GELU FFN
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
2023  LLaMA 2 / 3, DeepSeek, Qwen, Mistral...
      Pre-Norm + RMSNorm + SwiGLU FFN
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.

---

## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.5. Exercise passed: you have understood this step.6. Read the values printed above and connect them to the concept in this cell.7. Read the values printed above and connect them to the concept in this cell.8. Read the values printed above and connect them to the concept in this cell.9. Read the values printed above and connect them to the concept in this cell.10. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.